In [1]:
import os
import requests
from dotenv import load_dotenv
# Load environment variables from .env file
load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["ALPHAVANTAGE_API_KEY"] = os.getenv("ALPHAVANTAGE_API_KEY")

In [2]:
# 1. Fetching Market Data
def fetch_market_data(stock_symbol: str) -> dict:
    """Simulate fetching stock market data for a given symbol."""
    url = 'https://www.alphavantage.co/query?function=OVERVIEW&symbol='+stock_symbol+'&apikey='+os.environ["ALPHAVANTAGE_API_KEY"]
    r = requests.get(url)
    data = r.json()
    del data['Description']
    del data['Address']
    del data['OfficialSite']
    return data

In [3]:
from datetime import datetime, timedelta

# 2. Sentiment Analysis
def analyze_sentiment(stock_symbol: str) -> dict:
    """Perform sentiment analysis on financial news for a stock."""
    currdate = datetime.now().strftime('%Y%m%dT%H%M')
    prevdate = (datetime.now() - timedelta(days=1)).strftime('%Y%m%dT%H%M')
    url = 'https://www.alphavantage.co/query?function=NEWS_SENTIMENT&tickers='+stock_symbol+'&sort=LATEST&time_from='+prevdate+'&time_to='+currdate+'&apikey='+os.environ["ALPHAVANTAGE_API_KEY"]
    r = requests.get(url)
    data = r.json()
    return data

In [4]:
def investment_strategy(stock_symbol, stock_data, news_feed):
    """Generate a recommendation based on the stock market data and news sentiment."""
    recommendations = []

    # Criteria 1: P/E Ratio
    pe_ratio = float(stock_data["PERatio"])
    if pe_ratio < 20:
        recommendations.append("Buy (Low P/E Ratio)")
    elif pe_ratio > 30:
        recommendations.append("Sell (High P/E Ratio)")

    # Criteria 2: Dividend Yield
    dividend_yield = float(stock_data["DividendYield"])
    if dividend_yield > 0.02:
        recommendations.append("Buy (High Dividend Yield)")
    elif dividend_yield < 0.01:
        recommendations.append("Sell (Low Dividend Yield)")

    # Criteria 3: Analyst Ratings
    strong_buy = int(stock_data["AnalystRatingStrongBuy"])
    buy = int(stock_data["AnalystRatingBuy"])
    hold = int(stock_data["AnalystRatingHold"])
    sell = int(stock_data["AnalystRatingSell"])
    strong_sell = int(stock_data["AnalystRatingStrongSell"])

    if strong_buy + buy > hold + sell + strong_sell:
        recommendations.append("Buy (Positive Analyst Ratings)")
    else:
        recommendations.append("Sell (Negative Analyst Ratings)")

    # Criteria 4: 52-Week High/Low
    current_price = float(stock_data["50DayMovingAverage"])
    week_52_high = float(stock_data["52WeekHigh"])
    week_52_low = float(stock_data["52WeekLow"])

    if current_price < (week_52_low + (week_52_high - week_52_low) * 0.25):
        recommendations.append("Buy (Near 52-Week Low)")
    elif current_price > (week_52_high - (week_52_high - week_52_low) * 0.25):
        recommendations.append("Sell (Near 52-Week High)")

    sentiment_scores = [item["ticker_sentiment_score"] for item in news_feed["feed"] if item["ticker_sentiment"][0]["ticker"] == stock_symbol]
    if sentiment_scores:
        avg_sentiment_score = sum(sentiment_scores) / len(sentiment_scores)
        if avg_sentiment_score >= 0.35:
            recommendations.append("Buy (Bullish News Sentiment)")
        elif avg_sentiment_score >= 0.15:
            recommendations.append("Buy (Somewhat-Bullish News Sentiment)")
        elif avg_sentiment_score <= -0.35:
            recommendations.append("Sell (Bearish News Sentiment)")
        elif avg_sentiment_score <= -0.15:
            recommendations.append("Sell (Somewhat-Bearish News Sentiment)")
    return f"Recommendation for {stock_symbol}: {recommendations}"

In [5]:
import os
from langchain_openai import ChatOpenAI
from langgraph_supervisor import create_supervisor
from langgraph.prebuilt import create_react_agent

# Initialize the Chat model
model = ChatOpenAI(model="gpt-4o")

### --- CREATE AGENTS --- ###

# Market Data Agent
market_data_expert = create_react_agent(
    model=model,
    tools=[fetch_market_data],
    name="market_data_expert",
    prompt="You are an expert in stock market data. Fetch stock data when requested."
)

# Sentiment Analysis Agent
sentiment_expert = create_react_agent(
    model=model,
    tools=[analyze_sentiment],
    name="sentiment_expert",
    prompt="You analyze financial news and social media sentiment for stock symbols."
)

# Investment Strategy Agent
strategy_expert = create_react_agent(
    model=model,
    tools=[investment_strategy],
    name="strategy_expert",
    prompt="You make investment recommendations based on market and news sentiment."
)

### --- SUPERVISOR AGENT --- ###

market_supervisor = create_supervisor(
    agents=[market_data_expert, sentiment_expert, strategy_expert],
    model=model,
    prompt=(
        "You are a financial market supervisor managing four expert agents: market data, sentiment, "
        "quantitative analysis, and investment strategy. For stock queries, use market_data_expert. "
        "For news/social sentiment, use sentiment_expert. For stock price analysis, use quant_expert. "
        "For final investment recommendations, use strategy_expert."
    )
)

# Compile into an executable workflow
app = market_supervisor.compile()

In [6]:
### --- RUN THE SYSTEM --- ###
stock_query = {
    "messages": [
        {"role": "user", "content": "What is the investment recommendation for AMZN?"}
    ]
}

# Execute query
result = app.invoke(stock_query)

print(result['messages'][-1].content)

I encountered an issue with retrieving all the necessary data to generate an investment recommendation for AMZN. Would you like to provide any specific assumptions or request additional data points?
